# Figure 1
5-year survival by stroke type. A: all patients. B: 90-day survivors.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

In [ ]:
data_path = '../data/excel_4figures.xlsx'
output_dir = '../data/figures'

DAYS_PER_YEAR = 365.25
FOLLOW_UP_DAYS = 5 * DAYS_PER_YEAR
LANDMARK_DAYS = 90

# same-day deaths
MIN_DURATION_DAYS = 0.1

# colors from original paper
STROKE_TYPES = {
    'Ischemic stroke': '#5a7a8c',
    'Intracerebral hemorrhage': '#b32f1e',
}

AT_RISK_YEARS = [0, 1, 2, 3, 4, 5]

In [ ]:
survival_df = pd.read_excel(data_path).dropna(how='all')
survival_df.shape

In [ ]:
# censored at 5 years
onset = pd.to_datetime(survival_df['Onset date'].astype(str), format='%Y%m%d')
death = pd.to_datetime(survival_df['5Y Death date'], format='%d/%m/%Y')
days_to_death = (death - onset).dt.days

survival_df['death_5y'] = days_to_death <= FOLLOW_UP_DAYS
survival_df['duration_years'] = (days_to_death.clip(lower=MIN_DURATION_DAYS)
                                 .where(survival_df['death_5y'], FOLLOW_UP_DAYS) / DAYS_PER_YEAR)

# from dates; Mortality_90d inconsistent
survival_df['survived_90d'] = ~(days_to_death <= LANDMARK_DAYS)

survivors_90d_df = survival_df[survival_df['survived_90d']]

survival_df.groupby('Type of event')[['death_5y', 'survived_90d']].agg(['size', 'sum'])

In [ ]:
def fit_km_per_type(cohort_df):
    fitters = {}
    for stroke_type in STROKE_TYPES:
        type_df = cohort_df[cohort_df['Type of event'] == stroke_type]
        kmf = KaplanMeierFitter(label=stroke_type)
        kmf.fit(type_df['duration_years'], event_observed=type_df['death_5y'])
        fitters[stroke_type] = kmf
    return fitters


def logrank_p_value(cohort_df):
    is_df, ich_df = (cohort_df[cohort_df['Type of event'] == stroke_type] for stroke_type in STROKE_TYPES)
    result = logrank_test(is_df['duration_years'], ich_df['duration_years'],
                          event_observed_A=is_df['death_5y'], event_observed_B=ich_df['death_5y'])
    return result.p_value


def format_p_value(p_value):
    if p_value < 0.001:
        return 'p < 0.001'
    return f'p = {p_value:.3f}'


all_fitters = fit_km_per_type(survival_df)
survivor_fitters = fit_km_per_type(survivors_90d_df)

all_p_value = logrank_p_value(survival_df)
survivor_p_value = logrank_p_value(survivors_90d_df)
all_p_value, survivor_p_value

In [ ]:
AT_RISK_HEADER_Y = -0.2
AT_RISK_ROW_SPACING = 0.055

# axes fraction
AT_RISK_LINE_X = (-0.12, -0.06)


def draw_at_risk_table(ax, fitters):
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    ax.text(AT_RISK_LINE_X[0], AT_RISK_HEADER_Y, 'Number at risk:', ha='left', transform=ax.transAxes)

    for row, (stroke_type, kmf) in enumerate(fitters.items(), start=1):
        y = AT_RISK_HEADER_Y - row * AT_RISK_ROW_SPACING
        ax.plot(AT_RISK_LINE_X, [y, y], color=STROKE_TYPES[stroke_type], transform=ax.transAxes, clip_on=False)

        for year in AT_RISK_YEARS:
            n_at_risk = (kmf.durations >= year).sum()
            ax.text(year, y, n_at_risk, ha='center', va='center', transform=trans)


def plot_km_panel(ax, fitters, panel_letter):
    for stroke_type, kmf in fitters.items():
        kmf.plot(ax=ax, color=STROKE_TYPES[stroke_type], ci_show=False)

    # remove upper and right spine
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_color('black')

    # Set axes
    ax.set_ylim(0, 1)
    ax.set_xlim(0, 5)
    ax.set_xticks(AT_RISK_YEARS)
    ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1])
    ax.set_yticklabels(['0', '20', '40', '60', '80', '100'])
    ax.yaxis.set_ticks_position('left')
    ax.set_ylabel('Survival rate (%)')
    ax.set_xlabel('Years')
    ax.grid(False)

    ax.legend(loc='lower right', frameon=False)
    ax.text(-0.15, 1.05, panel_letter, fontsize=14, fontweight='bold', transform=ax.transAxes)

    draw_at_risk_table(ax, fitters)

In [ ]:
fig, (all_ax, survivor_ax) = plt.subplots(1, 2, figsize=(12, 5.5))

# A: all patients, B: 90-day survivors
plot_km_panel(all_ax, all_fitters, 'A')
plot_km_panel(survivor_ax, survivor_fitters, 'B')

# room for at-risk tables
plt.subplots_adjust(left=0.08, right=0.97, bottom=0.33, wspace=0.3)

In [ ]:
os.makedirs(output_dir, exist_ok=True)
fig.savefig(os.path.join(output_dir, 'figure1_kaplan_meier_5y.pdf'), dpi=600, bbox_inches='tight')
fig.savefig(os.path.join(output_dir, 'figure1_kaplan_meier_5y.png'), dpi=300, bbox_inches='tight')